# Time Series Analysis: ARIMA vs VAR Models

Using **real** UCI Air Quality data (hourly pollutant readings from a sensor in Italy, March 2004 – April 2005).

Source: https://archive.ics.uci.edu/dataset/360/air+quality

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.vector_ar.var_model import VAR
from statsmodels.tsa.stattools import adfuller
from sklearn.metrics import mean_squared_error, mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 5)

## 1. Load & Prepare Real Data

In [ ]:
# The CSV uses semicolons, European decimals (comma), and dots in time (18.00.00)
df_raw = pd.read_csv(
    'AirQualityUCI.csv',
    sep=';',
    decimal=',',
    na_values=-200,
)

# Drop trailing empty columns and rows with all NaN
df_raw = df_raw.drop(columns=[c for c in df_raw.columns if 'Unnamed' in c or c == ''], errors='ignore')
df_raw = df_raw.dropna(how='all').reset_index(drop=True)

# Combine Date + Time into a proper DatetimeIndex (time uses dots: 18.00.00)
df_raw['Datetime'] = pd.to_datetime(
    df_raw['Date'] + ' ' + df_raw['Time'].str.replace('.', ':'),
    format='%d/%m/%Y %H:%M:%S',
)

# Select a few key pollutants
cols = ['CO(GT)', 'C6H6(GT)', 'NO2(GT)', 'T']  # CO, Benzene, NO2, Temperature
df = df_raw[['Datetime'] + cols].copy()
df = df.dropna(subset=cols)
df = df.set_index('Datetime')

# Resample to daily averages to reduce noise and speed up fitting
df = df.resample('D').mean().dropna()

print(f'Shape: {df.shape}')
print(f'Date range: {df.index[0].date()} to {df.index[-1].date()}')
print(f'Columns: {list(df.columns)}')
df.head(10)

In [ ]:
# Plot
colors = ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0']
fig, axes = plt.subplots(len(cols), 1, figsize=(14, 3.5*len(cols)), sharex=True)
for i, col in enumerate(cols):
    axes[i].plot(df.index, df[col], color=colors[i], linewidth=1.2)
    axes[i].set_title(col, fontsize=13)
    axes[i].set_ylabel('Value')
    axes[i].grid(True, alpha=0.3)
plt.suptitle('UCI Air Quality — Daily Averages', fontsize=15, y=1.01)
plt.tight_layout()
plt.show()

## 2. Train-Test Split

In [ ]:
split_ratio = 0.8
split_idx = int(len(df) * split_ratio)

train = df.iloc[:split_idx]
test  = df.iloc[split_idx:]

print(f'Total: {len(df)} days')
print(f'Train: {len(train)} days  ({train.index[0].date()} → {train.index[-1].date()})')
print(f'Test:  {len(test)} days  ({test.index[0].date()} → {test.index[-1].date()})')

## 3. Stationarity Test (ADF)

In [ ]:
print(f'{"Variable":<12} {"p-value":>10} {"Status":>16}')
print('-' * 42)
for col in cols:
    p = adfuller(train[col], autolag='AIC')[1]
    status = 'Stationary' if p < 0.05 else 'Non-stationary'
    print(f'{col:<12} {p:>10.4f} {status:>16}')

## 4. ARIMA Model

Univariate — each pollutant modeled independently.
Grid search over (p, d, q) to minimize AIC.

In [ ]:
arima_preds = {}

for col in cols:
    best_aic = np.inf
    best_order = (1, 1, 1)
    for p in [0, 1, 2, 3]:
        for d in [0, 1]:
            for q in [0, 1, 2, 3]:
                try:
                    fitted = ARIMA(train[col], order=(p, d, q)).fit()
                    if fitted.aic < best_aic:
                        best_aic = fitted.aic
                        best_order = (p, d, q)
                except:
                    pass
    
    fitted = ARIMA(train[col], order=best_order).fit()
    arima_preds[col] = fitted.forecast(steps=len(test)).values
    print(f'{col}: ARIMA{best_order}, AIC={best_aic:.2f}')

In [ ]:
fig, axes = plt.subplots(len(cols), 1, figsize=(14, 3.5*len(cols)))
for i, col in enumerate(cols):
    axes[i].plot(train.index, train[col], label='Train', color=colors[i], linewidth=1.2)
    axes[i].plot(test.index, test[col].values, label='Actual', color='green', linewidth=2)
    axes[i].plot(test.index, arima_preds[col], label='ARIMA', color='red', linewidth=2, linestyle='--')
    axes[i].set_title(f'ARIMA — {col}', fontsize=13)
    axes[i].legend()
    axes[i].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. VAR Model

Multivariate — models all pollutants together, capturing cross-correlations.

In [ ]:
# Select optimal lag
lag_aics = {}
for lag in range(1, 16):
    try:
        result = VAR(train).fit(lag)
        lag_aics[lag] = result.aic
    except:
        break

lag_df = pd.DataFrame.from_dict(lag_aics, orient='index', columns=['AIC'])
optimal_lag = lag_df['AIC'].idxmin()
print('Lag AIC values:')
print(lag_df.round(2).to_string())
print(f'\nOptimal lag: {optimal_lag}')

In [ ]:
var_fitted = VAR(train).fit(optimal_lag)
var_forecast = var_fitted.forecast(train.values[-optimal_lag:], steps=len(test))

var_preds = {col: var_forecast[:, i] for i, col in enumerate(cols)}
print(f'VAR({optimal_lag}) fitted. AIC={var_fitted.aic:.2f}')

In [ ]:
fig, axes = plt.subplots(len(cols), 1, figsize=(14, 3.5*len(cols)))
for i, col in enumerate(cols):
    axes[i].plot(train.index, train[col], label='Train', color=colors[i], linewidth=1.2)
    axes[i].plot(test.index, test[col].values, label='Actual', color='green', linewidth=2)
    axes[i].plot(test.index, var_preds[col], label='VAR', color='orange', linewidth=2, linestyle='--')
    axes[i].set_title(f'VAR — {col}', fontsize=13)
    axes[i].legend()
    axes[i].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Side-by-Side: ARIMA vs VAR

In [ ]:
fig, axes = plt.subplots(len(cols), 1, figsize=(14, 3.5*len(cols)))
for i, col in enumerate(cols):
    axes[i].plot(test.index, test[col].values, label='Actual', color='green', linewidth=2)
    axes[i].plot(test.index, arima_preds[col], label='ARIMA', color='red', linewidth=2, linestyle='--')
    axes[i].plot(test.index, var_preds[col], label='VAR', color='orange', linewidth=2, linestyle='--')
    axes[i].set_title(f'ARIMA vs VAR — {col}', fontsize=13)
    axes[i].legend(fontsize=11)
    axes[i].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Evaluation Metrics

| Metric | Meaning |
|--------|---------|
| **MSE** | Mean Squared Error |
| **RMSE** | Root MSE (same unit as data) |
| **MAE** | Mean Absolute Error |
| **MAPE** | Mean Absolute Percentage Error (%) |

In [ ]:
def calc_metrics(y_true, y_pred):
    y_true = np.array(y_true).flatten()
    y_pred = np.array(y_pred).flatten()
    mse = mean_squared_error(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / np.where(y_true == 0, 1, y_true))) * 100
    return {
        'MSE': round(mse, 4),
        'RMSE': round(np.sqrt(mse), 4),
        'MAE': round(mean_absolute_error(y_true, y_pred), 4),
        'MAPE (%)': round(mape, 4)
    }

metric_names = ['MSE', 'RMSE', 'MAE', 'MAPE (%)']
arima_metrics = {}
var_metrics = {}

print('=' * 60)
for col in cols:
    actual = test[col].values
    arima_metrics[col] = calc_metrics(actual, arima_preds[col])
    var_metrics[col]   = calc_metrics(actual, var_preds[col])
    
    print(f'\n{col}:')
    print(f'{"Metric":<12} {"ARIMA":>12} {"VAR":>12} {"Better":>10}')
    print('-' * 48)
    for m in metric_names:
        a = arima_metrics[col][m]
        v = var_metrics[col][m]
        better = 'ARIMA' if a < v else 'VAR'
        print(f'{m:<12} {a:>12.4f} {v:>12.4f} {better:>10}')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
axes = axes.flatten()

for i, metric in enumerate(metric_names):
    x = np.arange(len(cols))
    w = 0.35
    a_vals = [arima_metrics[c][metric] for c in cols]
    v_vals = [var_metrics[c][metric] for c in cols]
    axes[i].bar(x - w/2, a_vals, w, label='ARIMA', color='#2196F3', alpha=0.8)
    axes[i].bar(x + w/2, v_vals, w, label='VAR', color='#FF9800', alpha=0.8)
    axes[i].set_ylabel(metric)
    axes[i].set_title(metric, fontsize=13)
    axes[i].set_xticks(x)
    axes[i].set_xticklabels(cols, rotation=15)
    axes[i].legend()
    axes[i].grid(True, alpha=0.3, axis='y')
plt.suptitle('ARIMA vs VAR — Metric Comparison', fontsize=15, y=1.01)
plt.tight_layout()
plt.show()

## 8. Overall Summary

In [ ]:
print('=' * 60)
print('OVERALL SUMMARY (average across all variables)')
print('=' * 60)

arima_wins = 0
var_wins = 0

print(f'\n{"Metric":<12} {"ARIMA (avg)":>15} {"VAR (avg)":>15} {"Better":>10}')
print('-' * 55)
for m in metric_names:
    a_avg = np.mean([arima_metrics[c][m] for c in cols])
    v_avg = np.mean([var_metrics[c][m] for c in cols])
    better = 'ARIMA' if a_avg < v_avg else 'VAR'
    if better == 'ARIMA':
        arima_wins += 1
    else:
        var_wins += 1
    print(f'{m:<12} {a_avg:>15.4f} {v_avg:>15.4f} {better:>10}')

print(f'\nARIMA wins {arima_wins}/{len(metric_names)} metrics')
print(f'VAR wins   {var_wins}/{len(metric_names)} metrics')
if arima_wins > var_wins:
    print('\n=> ARIMA performs better overall.')
elif var_wins > arima_wins:
    print('\n=> VAR performs better overall.')
else:
    print('\n=> Both models perform similarly.')

## 9. Residual Analysis

In [ ]:
fig, axes = plt.subplots(len(cols), 2, figsize=(14, 3.5*len(cols)))
for i, col in enumerate(cols):
    arima_resid = test[col].values - arima_preds[col]
    var_resid   = test[col].values - var_preds[col]
    axes[i, 0].plot(test.index, arima_resid, color='red', alpha=0.7)
    axes[i, 0].axhline(0, color='black', linestyle='--')
    axes[i, 0].set_title(f'{col} — ARIMA Residuals')
    axes[i, 0].grid(True, alpha=0.3)
    axes[i, 1].plot(test.index, var_resid, color='orange', alpha=0.7)
    axes[i, 1].axhline(0, color='black', linestyle='--')
    axes[i, 1].set_title(f'{col} — VAR Residuals')
    axes[i, 1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 10. Key Takeaways

| | ARIMA | VAR |
|---|---|---|
| **Type** | Univariate | Multivariate |
| **Captures cross-correlations** | No | Yes |
| **Parameters** | (p, d, q) per series | Lag order for all series |
| **Best for** | Independent series | Correlated series |

- **ARIMA**: each pollutant is modeled independently; good when series have strong own dynamics.
- **VAR**: captures how pollutants influence each other (e.g., CO and NO₂ both come from traffic emissions); can benefit from shared patterns but also risks noise from weak correlations.